In [62]:
import pandas as pd
import huggingface_hub
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling
from sklearn.model_selection import train_test_split
import os
import numpy as np

os.environ["http_proxy"] = "http://127.0.0.1:10809"
os.environ["https_proxy"] = "http://127.0.0.1:10809"



In [ ]:
# Upload dataset to huggingface
!huggingface-cli lfs-enable-largefiles

In [ ]:
login(token = "HF_TOKEN_REMOVED", add_to_git_credential=True)

### The repo is already created and do not need to be created again

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="soc-model")

### Upload the large file to the repo using Git LFS

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="E:/Data/job_posting/processed/finetune/df_titleLabel.csv",
    path_in_repo="df_titleLabel.csv",
    repo_id="Zexuan/soc_data",
    repo_type="dataset",
)

### Finetune a model

In [ ]:
train_df, test_df = train_test_split(pd.read_csv('F:/Data/job_posting/processed/finetune/df_titleLabel.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore'), test_size=0.3, random_state=42)

In [50]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, AutoModelForMaskedLM
import torch
from transformers import BertForSequenceClassification

In [51]:
#model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese")
model = BertForSequenceClassification.from_pretrained("bert-base-chinese")

Some weights of the model checkpoint at bert-base-chinese were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [56]:
#tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
# Tokenize the text and convert it into input features
train_texts = str(train_df_sample['工作描述'].tolist())
train_labels = str(train_df_sample['soc_code'].tolist())
train_encodings = tokenizer(train_texts, truncation=True, padding=True)

test_texts = str(test_df_sample['工作描述'].tolist())
test_labels = str(test_df_sample['soc_code'].tolist())
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [69]:
import torch
from transformers import AdamW
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    warmup_steps=0,                  # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    load_best_model_at_end=True,
    evaluation_strategy="steps",
    eval_steps=500,
)

optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = get_scheduler("linear", optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*3)

def compute_metrics(eval_pred, eval_labels):
    preds = np.argmax(eval_pred, axis=1)
    return {"accuracy": (preds == eval_labels).mean()}

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    optimizer=optimizer,
    lr_scheduler=scheduler,
    compute_metrics=compute_metrics,
)

trainer.train()



TypeError: __init__() got an unexpected keyword argument 'optimizer'

In [71]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)


from transformers import AdamW
from transformers import get_scheduler

optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = get_scheduler("linear", optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*3)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    optimizer=optimizer,
    lr_scheduler=scheduler,
    compute_metrics=compute_metrics,
    num_train_epochs=3,
    batch_size=16,
)

trainer.train()

TypeError: __init__() got an unexpected keyword argument 'optimizer'